In [40]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import kagglehub
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

In [41]:
path = kagglehub.dataset_download("abhi8923shriv/sentiment-analysis-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/legion/.cache/kagglehub/datasets/abhi8923shriv/sentiment-analysis-dataset/versions/9


In [42]:
import os
os.listdir(path)

['testdata.manual.2009.06.14.csv',
 'test.csv',
 'training.1600000.processed.noemoticon.csv',
 'train.csv']

In [43]:
train_path = path + '/train.csv'

In [44]:
df = pd.read_csv(train_path, encoding = 'latin1', on_bad_lines = 'skip').dropna()

In [45]:
df = df[['text', 'sentiment']]

In [46]:
encoder = LabelEncoder()

In [47]:
df['sentiment'] = encoder.fit_transform(df['sentiment'])

In [48]:
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13875.96it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
device = torch.device('cuda')

In [50]:
tensors = bert_tokenizer(df['text'].tolist(), padding = 'longest', max_length = 128, truncation = True, return_tensors = 'pt')

In [51]:
class SentimentDataset(Dataset):
  def __init__(self, encodings, labels):
    self.input_ids =  encodings['input_ids']
    self.token_type_ids = encodings['token_type_ids']
    self.attention_mask = encodings['attention_mask']
    self.labels = torch.tensor(labels)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return self.input_ids[idx],self.token_type_ids[idx], self.attention_mask[idx], self.labels[idx]

In [52]:
test_ds = SentimentDataset(tensors, df['sentiment'].values)

In [53]:
test_dataloader = DataLoader(test_ds, batch_size = 32, shuffle = True)

In [69]:
class SentimentModel(nn.Module):
    def __init__(self, bert_model):
        super().__init__()

        self.bert = bert_model
        self.classifier = nn.Sequential(
           nn.Linear(768, 3))

    def forward(self, input_ids, token_type_ids, attention_mask):
      out  = self.bert(input_ids = input_ids, token_type_ids = token_type_ids, attention_mask = attention_mask)['pooler_output']
      out = self.classifier(out)
      return out

In [70]:
sentiment_model = SentimentModel(bert_model)

In [71]:
sentiment_model = sentiment_model.to(device)

In [72]:
for p in sentiment_model.parameters():
  p.requires_grad = False

In [73]:
sentiment_model.classifier  = sentiment_model.classifier.requires_grad_(True)

In [74]:
epochs = 10

In [75]:
  loss_fn = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(sentiment_model.classifier.parameters(), lr=1e-4)

  for epoch in range(epochs):
    sum_loss = 0

    for input_ids, token_type_ids, attention_mask, label in tqdm(test_dataloader):
        input_ids, token_type_ids, attention_mask, label = input_ids.to(device), token_type_ids.to(device), attention_mask.to(device), label.to(device)
        preds = sentiment_model(input_ids, token_type_ids, attention_mask)

        loss = loss_fn(preds, label)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        sum_loss += loss.item()

    print(f"Total Loss: {sum_loss /(len(test_dataloader))}")

  0%|          | 0/859 [00:00<?, ?it/s]

100%|██████████| 859/859 [02:11<00:00,  6.55it/s]


Total Loss: 1.0663901748562858


100%|██████████| 859/859 [02:09<00:00,  6.65it/s]


Total Loss: 1.0312391546607989


...

Total Loss: 0.9341799096400579


100%|██████████| 859/859 [02:08<00:00,  6.69it/s]


Total Loss: 0.9253523001432141


100%|██████████| 859/859 [02:05<00:00,  6.87it/s]

Total Loss: 0.9184475277438846


In [78]:
torch.save(sentiment_model.state_dict(), 'model_parameters.pt')